# **딥러닝 맛보기**

# 1.환경준비

## (1) 라이브러리 불러오기

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import  train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import  StandardScaler, MinMaxScaler

In [2]:
import torch
from torch import nn
from torch.utils.data import  DataLoader, TensorDataset
from torch.optim import Adam

## (2) 필요 함수 생성

- 모델링을 위한 데이터로더 만들기
    - 학습시, 배치 단위로 데이터를 처리하기 위함

In [9]:
def make_DataSet(X_train, X_val, y_train, y_val, batch_size = 32) :
    # batch_size : 모델이 한 번에 몇 개의 데이터를 보고 학습할지 정하는 값

    # 데이터 텐서로 변환
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)

    # TensorDataset 생성 : 텐서 데이터셋으로 합치기
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

    # DataLoader 생성
    train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)

    return train_loader, X_val_tensor, y_val_tensor

- 학습을 위한 함수

In [10]:
def train(dataloader, model, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    tr_loss = 0
    model.train()                                           # 훈련 모드로 설정
    for batch, (X, y) in enumerate(dataloader):             # batch : 현재 배치 번호, (X,y): 입력 데이터와 레이블
        X, y = X.to(device), y.to(device)

        # Feed Forward
        pred = model(X)
        loss = loss_fn(pred, y)
        tr_loss += loss

        # Backpropagation
        loss.backward()                     # 역전파를 통해 모델의 각 파라미터에 대한 손실의 기울기를 계산
        optimizer.step()                    # 옵티마이저가 계산된 기울기를 사용하여 모델의 파라미터를 업데이트
        optimizer.zero_grad()               # 옵티마이저의 기울기 값 초기화. 기울기가 누적되는 것 방지.

    tr_loss /= num_batches                  # 모든 배치에서의 loss 평균

    return tr_loss.item()

- 검증을 위한 함수

In [11]:
def evaluate(X_val_tensor, y_val_tensor, model, loss_fn, device):
    model.eval()                                                # 모델을 평가 모드로 설정

    with torch.no_grad():                                       # 평가 과정에서 기울기를 계산하지 않도록 설정
        X, y = X_val_tensor.to(device), y_val_tensor.to(device)
        pred = model(X)
        eval_loss = loss_fn(pred, y).item()                     # 예측 값 pred와 실제 값 y 사이의 손실 계산

    return eval_loss

## (3) Device 준비(cpu or gpu)

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f'{device} available')

cuda available


# 2.Regression : Advertising

## (1) 데이터 전처리


### 1) 데이터 준비

In [13]:
path = 'advertising.csv'
advertising = pd.read_csv(path)
advertising.head()

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,9.3
3,151.5,41.3,58.5,18.5
4,180.8,10.8,58.4,12.9


In [14]:
target = 'Sales'
X = advertising.drop(target, axis=1)
y = advertising[target]

### 2) 가변수화

### 3) 데이터분할

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

### 4) 스케일링

In [16]:
# 학습 데이터 기준으로 스케일링 기준을 만들고, 검증 데이터에는 그 기준만 적용.
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## (2) 딥러닝 모델링

### 1) 딥러닝을 위한 데이터 준비

*  pandas 데이터프레임 ==> PyTorch의 DataLoader로 변환
    * 데이터 텐서로 변환
    * 텐서 데이터셋으로 합치기 : x, y
    * 데이터 로더 생성

* 1-(2) 에서 생성한 함수 : **make_DataLoader**

In [17]:
train_loader, X_test_ts, y_test_ts = make_DataSet(X_train, X_test, y_train, y_test, batch_size = 4)

In [18]:
# 첫번째 배치만 로딩해서 살펴보기
for x, y in train_loader:
    print(x.shape)
    print(y.shape)
    print(y.dtype)

torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([4, 3])
torch.Size([4, 1])
torch.float32
torch.Size([

### 2) 모델 선언

In [19]:
n_feature = X.shape[1]

# 모델 구조 설계
model = nn.Sequential(
            nn.Linear(n_feature, 3),    # hidden layer(input, output)
            nn.ReLU(),                  # 활성화 함수
            nn.Linear(3,1)              # output layer
            ).to(device)                # cpu/gpu 사용 설정

print(model)

Sequential(
  (0): Linear(in_features=3, out_features=3, bias=True)
  (1): ReLU()
  (2): Linear(in_features=3, out_features=1, bias=True)
)


* Loss function과 Optimizer

In [20]:
loss_fn = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=0.001)   # model.parameters() : 모델의 가중치와 편향

### 4) 학습

* 1-(2)에서 생성한 함수 : **train**, **evaluate**

In [21]:
epochs = 30
for t in range(epochs):
    tr_loss = train(train_loader, model, loss_fn, optimizer, device)
    test_loss = evaluate(X_test_ts, y_test_ts, model, loss_fn, device)

    print(f"Epoch {t+1}, train loss : {tr_loss:4f}, test_loss : {test_loss:4f}")

Epoch 1, train loss : 215.660934, test_loss : 232.857330
Epoch 2, train loss : 211.993408, test_loss : 229.044785
Epoch 3, train loss : 208.422882, test_loss : 225.216782
Epoch 4, train loss : 204.804642, test_loss : 221.180618
Epoch 5, train loss : 200.988220, test_loss : 216.919388
Epoch 6, train loss : 196.961716, test_loss : 212.327637
Epoch 7, train loss : 192.674759, test_loss : 207.488678
Epoch 8, train loss : 188.124374, test_loss : 202.420731
Epoch 9, train loss : 183.331451, test_loss : 196.996811
Epoch 10, train loss : 178.242828, test_loss : 191.412384
Epoch 11, train loss : 172.948837, test_loss : 185.542801
Epoch 12, train loss : 167.455246, test_loss : 179.332306
Epoch 13, train loss : 161.704834, test_loss : 173.085663
Epoch 14, train loss : 155.835068, test_loss : 166.647827
Epoch 15, train loss : 149.842697, test_loss : 159.997421
Epoch 16, train loss : 143.714523, test_loss : 153.209396
Epoch 17, train loss : 137.511658, test_loss : 146.394409
Epoch 18, train loss : 

### 5) 모델 평가

In [23]:
evaluate(X_test_ts, y_test_ts, model, loss_fn, device)

64.25133514404297

# 3.Regression : 보스턴 집값

## (1) 데이터 전처리 - 기본

### 1) 데이터 준비

In [26]:
path = 'boston.csv'
df = pd.read_csv(path)
df.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,5.33,36.2


In [27]:
target = 'medv'
X = df.drop(target, axis=1)
y = df[target]

### 2) 가변수화

### 3) 데이터분할

In [28]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=300)

### 4) 스케일링

In [29]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

## (2) 딥러닝 모델링

### 1) 딥러닝을 위한 데이터 준비

*  pandas 데이터프레임 ==> PyTorch의 DataLoader로 변환
    * 데이터 텐서로 변환
    * 텐서 데이터셋으로 합치기 : x, y
    * 데이터 로더 생성

* 1-(2) 에서 생성한 함수 : **make_DataLoader**

In [30]:
train_loader, X_val_ts, y_val_ts = make_DataSet(X_train, X_val, y_train, y_val, batch_size=4)

### 2) 모델 선언

In [31]:
n_feature = X.shape[1]

# 모델 구조 설계
model = nn.Sequential(
    nn.Linear(n_feature, 16),       # 입력층 -> 은닉층
    nn.ReLU(),
    nn.Linear(16,8),                # 은닉층 -> 은닉층
    nn.ReLU(),
    nn.Linear(8,1)                  # 출력층
).to(device)

print(model)

Sequential(
  (0): Linear(in_features=12, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=1, bias=True)
)


* Loss function과 Optimizer

In [32]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### 4) 학습

In [33]:
epochs = 50
for t in range(epochs):
    tr_loss = train(train_loader, model, loss_fn, optimizer, device)
    val_loss = evaluate(X_val_ts, y_val_ts, model, loss_fn, device)

    print(f"Epoch {t+1}, train loss : {tr_loss:.4f}, val loss : {val_loss:.4f}")

Epoch 1, train loss : 603.0211, val loss : 512.9906
Epoch 2, train loss : 535.9155, val loss : 385.5377
Epoch 3, train loss : 320.3020, val loss : 163.3085
Epoch 4, train loss : 159.5754, val loss : 125.7802
Epoch 5, train loss : 128.0264, val loss : 107.5763
Epoch 6, train loss : 104.5974, val loss : 85.1674
Epoch 7, train loss : 84.3247, val loss : 71.5460
Epoch 8, train loss : 70.8435, val loss : 59.4202
Epoch 9, train loss : 61.8002, val loss : 52.8934
Epoch 10, train loss : 56.4098, val loss : 48.9934
Epoch 11, train loss : 52.6291, val loss : 47.0547
Epoch 12, train loss : 50.0866, val loss : 43.5770
Epoch 13, train loss : 47.4451, val loss : 41.6022
Epoch 14, train loss : 45.2957, val loss : 39.8812
Epoch 15, train loss : 43.0645, val loss : 38.2333
Epoch 16, train loss : 40.9871, val loss : 36.4835
Epoch 17, train loss : 38.5499, val loss : 35.7216
Epoch 18, train loss : 36.6905, val loss : 33.5071
Epoch 19, train loss : 34.8990, val loss : 32.1141
Epoch 20, train loss : 33.240

### 5) 모델 평가

In [34]:
evaluate(X_val_ts, y_val_ts, model, loss_fn, device)

19.90017318725586